In [13]:
import pandas as pd
train_df = pd.read_parquet(r"final_data\chunk-based-split-3\no_scaler_train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split-3\no_scaler_valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split-3\no_scaler_test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [2]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        # Tối ưu RAM: Lấy column features một lần, ép kiểu thẳng qua numpy float32 rồi chiếu qua as_tensor
        feat_cols = [col for col in df.columns if col != 'label']
        self.X = torch.as_tensor(df[feat_cols].to_numpy(dtype=np.float32))
        self.y = torch.as_tensor(df['label'].to_numpy(dtype=np.int64))
        
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        all_indices = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels = self.y[all_indices + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        classes = np.unique(window_labels)
        
        for c in classes:
            c_indices = all_indices[window_labels == c]
            count = len(c_indices)
            
            if count > max_samples_per_class:
                c_indices = np.random.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            
        np.random.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices)
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y


In [14]:
class DynamicUndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, 
                 step=5, resample_each_epoch=False):
        feat_cols = [col for col in df.columns if col != 'label']
        self.X = torch.as_tensor(df[feat_cols].to_numpy(dtype=np.float32))
        self.y = torch.as_tensor(df['label'].to_numpy(dtype=np.int64))
        
        self.time_steps = time_steps
        self.step = step
        self.max_samples_per_class = max_samples_per_class
        self.resample_each_epoch = resample_each_epoch
        
        all_indices = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels = self.y[all_indices + self.time_steps - 1].numpy()
        
        self.class_indices = {}
        for c in np.unique(window_labels):
            self.class_indices[c] = all_indices[window_labels == c]
            print(f"Class {c}: {len(self.class_indices[c])} cửa sổ")
        
        self._resample()
    
    def _resample(self):
        self.valid_indices = []
        for c, c_indices in self.class_indices.items():
            count = len(c_indices)
            if count > self.max_samples_per_class:
                sampled = np.random.choice(
                    c_indices, self.max_samples_per_class, replace=False
                )
            else:
                sampled = c_indices
            self.valid_indices.extend(sampled)
        np.random.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices)
    
    def on_epoch_start(self):
        if self.resample_each_epoch:
            self._resample()

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        return window_X, label_y 

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_df.shape[1] - 1 
NUM_CLASSES = len(train_df['label'].unique())
TIME_STEPS = 10


# CƠ CHẾ ATTENTION THEO THỜI GIAN (Temporal Attention)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# ĐẠI TU MODEL CNN-BiLSTM
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        x = x.permute(0, 2, 1) 
        
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        

        out = self.layer_norm(out)
        
        context_vector, attn_weights = self.attention(out)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {DEVICE}")

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
print(model)

Đang sử dụng thiết bị: cuda
CNN_BiLSTM_Attention(
  (res1): ResidualBlock1D(
    (conv1): Conv1d(314, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (relu): ReLU()
    (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (dropout): Dropout1d(p=0.2, inplace=False)
    (se): SEBlock1D(
      (avg_pool): AdaptiveAvgPool1d(output_size=1)
      (fc): Sequential(
        (0): Linear(in_features=64, out_features=8, bias=False)
        (1): ReLU(inplace=True)
        (2): Linear(in_features=8, out_features=64, bias=False)
        (3): Sigmoid()
      )
    )
    (shortcut): Sequential(
      (0): Conv1d(314, 64, kernel_size=(1,), stride=(1,))
      (1): GroupNorm(8, 64, eps=1e-05, affine=True)
    )
  )
  (res2): ResidualBlock1D(
    (conv1): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (relu):

In [16]:
import torch
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
import os
import numpy as np

def train_bi_lstm_cnn(model, train_loader, val_loader, device, epochs=30, patience=5, save_path="model_final/best_cnn_bilstm_oof.pth"):
    """
    Hàm đóng gói quá trình train mô hình CNN-BiLSTM.
    Thích hợp để gọi lại nhiều lần trong vòng lặp K-Fold OOF.
    """
    # Đảm bảo thư mục lưu model tồn tại
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Dùng Cross Entropy với Label Smoothing = 0.1
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Dùng Adam với weight decay
    optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

    # Giảm learning rate nếu val loss không cải thiện
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        total_train = 0
        
        all_train_preds = []
        all_train_targets = []
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for inputs, labels in loop:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            # Gradient Clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            
            # Tối ưu RAM: Dùng append khối numpy thay vì phân rã list bằng extend
            all_train_preds.append(predicted.cpu().numpy())
            all_train_targets.append(labels.cpu().numpy())
            
            loop.set_postfix(loss=loss.item())

        avg_train_loss = train_loss / total_train
        train_macro_f1 = f1_score(np.concatenate(all_train_targets), np.concatenate(all_train_preds), average='macro')
        
        model.eval()
        val_loss = 0.0
        total_val = 0
        
        all_val_preds = []
        all_val_targets = []
        
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Validation]", leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                
                total_val += labels.size(0)
                
                # Tối ưu RAM: Dùng append
                all_val_preds.append(predicted.cpu().numpy())
                all_val_targets.append(labels.cpu().numpy())

        avg_val_loss = val_loss / total_val
        val_macro_f1 = f1_score(np.concatenate(all_val_targets), np.concatenate(all_val_preds), average='macro')
        
        print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.4f}, Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
        
        scheduler.step(avg_val_loss)
        
        # Lưu model nếu tốt nhất
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n[Early Stopping] Model không cải thiện sau {patience} epochs. Dừng huấn luyện.")
                break

    # Load lại model chứa trọng số tốt nhất trước khi trả về
    print(f"\n[Thông báo] Khôi phục trọng số tốt nhất từ: {save_path}")
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    
    return model


In [17]:
def predict_bi_lstm_cnn(model, data_loader, device):
    """
    Hàm dự đoán xác suất (predict_proba) từ dataloader thông qua model.
    Trả về probabilities hữu dụng để làm Meta-features.
    """
    model.eval()
    all_probs = []
    
    with torch.no_grad():
        for inputs, _ in tqdm(data_loader, desc="[Đang trích xuất đặc trưng OOF]", leave=False):
            inputs = inputs.to(device)
            # Truyền mảng qua model
            outputs = model(inputs)
            
            # Tính softmax để ra phân phối xác suất
            probs = torch.softmax(outputs, dim=1)
            
            # Tối ưu RAM: Lưu theo list block thay vì xé rác bộ nhớ bằng extend mảnh nhỏ
            all_probs.append(probs.cpu().numpy())
            
    import numpy as np
    # Nối block siêu tốc
    return np.vstack(all_probs)



In [18]:
class SequentialSlidingWindowDataset(Dataset):
    """
    Dataset dùng riêng cho việc Validation, Test, và sinh OOF Features.
    Không Shuffle, không Undersample, Step trượt = 1.
    Giữ đúng thứ tự dòng để sau này khớp kết quả dự đoán với DataFrame gốc.
    """
    def __init__(self, df, time_steps):
        # Tối ưu RAM: Lấy column features một lần, ép kiểu thẳng qua numpy float32 rồi chiếu qua as_tensor
        feat_cols = [col for col in df.columns if col != 'label']
        self.X = torch.as_tensor(df[feat_cols].to_numpy(dtype=np.float32))
        self.y = torch.as_tensor(df['label'].to_numpy(dtype=np.int64))
        
        self.time_steps = time_steps
        
        # Step = 1, không trộn, quét qua từng dòng một
        self.valid_indices = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        
        print(f"Khởi tạo Sequential Dataset:")
        print(f"- Chiều dài dữ liệu gốc: {len(self.X)}")
        print(f"- Kích thước cửa sổ (time_steps): {self.time_steps}")
        print(f"- Số lượng mẫu (windows) sinh ra: {len(self.valid_indices)}")
        print(f"-> Mẫu đầu tiên tương ứng với dòng index {self.time_steps - 1} của DataFrame.")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # Lấy theo thứ tự tuần tự
        actual_idx = self.valid_indices[idx]
        
        # Cắt tensor X
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        
        # Lấy nhãn của dòng cuối cùng trong cửa sổ
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y


In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import QuantileTransformer
from torch.utils.data import DataLoader
import os

def get_stratified_tscv_indices(y, n_splits=5):
    """
    Hàm chia Stratified TimeSeriesSplit:
    Đảm bảo lấy các phần tử xuất hiện sớm nhất của TỪNG CLASS để luyện tập.
    Sau đó gộp lại và sort theo thời gian (giúp duy trì tính chất chuỗi theo thời gian thực tế).
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    train_folds = [[] for _ in range(n_splits)]
    val_folds = [[] for _ in range(n_splits)]
    
    # Lặp qua từng class
    classes = np.unique(y)
    for c in classes:
        # Lấy list index của class c (đã được sắp xếp theo thời gian do train_df gốc đã sort)
        c_indices = np.where(y == c)[0]
        
        # Cắt TimeSeriesSplit trên nội bộ class c
        for fold, (tr_idx_rel, val_idx_rel) in enumerate(tscv.split(c_indices)):
            train_folds[fold].extend(c_indices[tr_idx_rel])
            val_folds[fold].extend(c_indices[val_idx_rel])
            
    # Hợp nhất và sắp xếp lại theo index ban đầu (thời gian) cho từng fold
    for fold in range(n_splits):
        train_folds[fold] = np.sort(train_folds[fold])
        val_folds[fold] = np.sort(val_folds[fold])
        
    return train_folds, val_folds

# --- GIAI ĐOẠN 1: TẠO OOF BẰNG CHUNK-BASED DEEP LEARNING ---
# Định nghĩa các tham số (giả sử TIME_STEPS trùng với tham số bạn đang thiết lập ở trên)
TIME_STEPS = 10 # Thay đổi giá trị này thành TIME_STEPS thực tế của bạn
BATCH_SIZE = 256
N_SPLITS = 3
NUM_CLASSES = len(train_df['label'].unique())
MAX_SAMPLES_TOTAL = 20000

# Khởi tạo ma trận lưu trữ đặc trưng OOF. 
# Dùng np.nan thay vì zeros để dễ nhận diện các dòng bị khuyết do giới hạn TIME_STEPS
meta_X_train_dl = np.full((len(train_df), NUM_CLASSES), np.nan)

# Sinh index Stratified TimeSeries
train_folds_idx, val_folds_idx = get_stratified_tscv_indices(train_df['label'].values, n_splits=N_SPLITS)

print("--- KHỞI ĐỘNG GIAI ĐOẠN 1: HUẤN LUYỆN 5 FOLDS CNN-BILSTM ĐỂ TẠO OOF ---")

for fold in range(N_SPLITS):
    print(f"\n{'='*20} FOLD {fold+1}/{N_SPLITS} {'='*20}")
    tr_idx = train_folds_idx[fold]
    val_idx = val_folds_idx[fold]
    
    # 1. Cắt DataFrame (dùng .copy() để tránh cảnh báo SettingWithCopy)
    fold_train_df = train_df.iloc[tr_idx].copy().reset_index(drop=True)
    fold_val_df = train_df.iloc[val_idx].copy().reset_index(drop=True)
    
    # 2. Fit và Transform bằng QuantileTransformer trên fold_train_df, Transform trên fold_val_df
    feat_cols = [col for col in fold_train_df.columns if col != 'label']
    scaler = QuantileTransformer(output_distribution="normal", random_state=42)
    
    print("Đang áp dụng QuantileTransformer...")
    fold_train_df[feat_cols] = scaler.fit_transform(fold_train_df[feat_cols])
    fold_val_df[feat_cols] = scaler.transform(fold_val_df[feat_cols])
    
    # 3. Tính toán giới hạn sample động theo từng fold
    # TimeSeriesSplit cấu trúc dữ liệu train lớn dần (1/5, 2/5, 3/5, 4/5, 5/5)
    current_max_samples = int(MAX_SAMPLES_TOTAL * (fold + 1) / N_SPLITS)
    print(f"Max samples giới hạn cho mỗi class tại Fold {fold+1}: {current_max_samples}")
    
    # 4. Tạo Dataset
    train_dataset = DynamicUndersampledSlidingWindowDataset(fold_train_df, time_steps=TIME_STEPS, max_samples_per_class=current_max_samples, step=3, resample_each_epoch= True)
    val_dataset = SequentialSlidingWindowDataset(fold_val_df, time_steps=TIME_STEPS)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # 5. Khởi tạo Model (Trọng số mới hoàn toàn cho mỗi fold)
    model_fold = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
    os.makedirs("model_oof", exist_ok=True)  # Đảm bảo thư mục lưu model tồn tại
    # 6. Huấn luyện (Early stopping dựa trên Val_Loader)
    save_model_path = f"model_oof/bilstm_oof_fold_{fold+1}.pth"
    model_fold = train_bi_lstm_cnn(
        model=model_fold, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        device=DEVICE, 
        epochs=30, 
        patience=12, 
        save_path=save_model_path
    )
    
    # 7. Sinh đặc trưng OOF từ tập Validation tuần tự
    fold_oof_preds = predict_bi_lstm_cnn(model=model_fold, data_loader=val_loader, device=DEVICE)
    
    # 8. MẢNH GHÉP QUAN TRỌNG: Gán OOF ngược lại vào meta_X_train_dl
    # Bỏ đi TIME_STEPS - 1 dòng đầu tiên của tập Validation (do Sliding Window không đọc được)
    actual_val_idx = val_idx[TIME_STEPS - 1 :]
    
    # Sanity Check
    if len(fold_oof_preds) == len(actual_val_idx):
        meta_X_train_dl[actual_val_idx, :] = fold_oof_preds
        print(f"> Fold {fold+1}: Đã ghép thành công {len(fold_oof_preds)} mẫu OOF vào meta_X_train_dl.")
    else:
        print(f"> [LỖI NGHIÊM TRỌNG]: Lệch kích thước! Dự đoán ({len(fold_oof_preds)}) != Index thực tế ({len(actual_val_idx)})")

print("\n--- HOÀN TẤT PHA 1: CHUYỂN ĐỔI THÀNH DATAFRAME VÀ LƯU TRỮ MA TRẬN ĐẶC TRƯNG OOF ---")

# Đưa Meta Features vào DataFrame, giữ nguyên index của train_df gốc để sau này dễ dàng `pd.concat`
dl_oof_columns = [f"dl_class_{i}" for i in range(NUM_CLASSES)]
meta_X_train_dl_df = pd.DataFrame(meta_X_train_dl, columns=dl_oof_columns, index=train_df.index)

# (Tùy chọn) Gắn thêm cột label thật để dễ đối chiếu sau này
meta_X_train_dl_df['label'] = train_df['label'].values

# Lưu ma trận Meta Features OOF xuống ổ cứng
os.makedirs("ip-files-train-meta", exist_ok=True)
oof_save_path = "ip-files-train-meta/meta_X_train_dl.parquet"
meta_X_train_dl_df.to_parquet(oof_save_path, engine="pyarrow")
print(f"Đã lưu Meta Features OOF của Deep Learning tại: {oof_save_path}")
print(f"Lưu ý: Sẽ có những dòng mang giá trị NaN ở các fold (do bị triệt tiêu TIME_STEPS - 1 dòng đầu).")

--- KHỞI ĐỘNG GIAI ĐOẠN 1: HUẤN LUYỆN 5 FOLDS CNN-BILSTM ĐỂ TẠO OOF ---

==================== FOLD 1/3 ====================
Đang áp dụng QuantileTransformer...
Max samples giới hạn cho mỗi class tại Fold 1: 6666
Class 0: 5374 cửa sổ
Class 1: 131102 cửa sổ
Class 2: 680 cửa sổ
Class 3: 9818 cửa sổ
Class 4: 5140 cửa sổ
Class 5: 663 cửa sổ
Class 6: 3208 cửa sổ
Class 7: 29096 cửa sổ
Class 8: 4535 cửa sổ
Class 9: 11242 cửa sổ
Class 10: 5027 cửa sổ


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Khởi tạo Sequential Dataset:
- Chiều dài dữ liệu gốc: 617658
- Kích thước cửa sổ (time_steps): 10
- Số lượng mẫu (windows) sinh ra: 617649
-> Mẫu đầu tiên tương ứng với dòng index 9 của DataFrame.


Epoch [1/30] - Train Loss: 1.1640, Macro F1: 0.7116 | Val Loss: 0.6098, Val Macro F1: 0.8511


Epoch [2/30] - Train Loss: 0.7523, Macro F1: 0.9405 | Val Loss: 0.5988, Val Macro F1: 0.8732


...

Epoch [10/30] - Train Loss: 0.6215, Macro F1: 0.9858 | Val Loss: 0.6869, Val Macro F1: 0.8379


Epoch [11/30] - Train Loss: 0.6121, Macro F1: 0.9891 | Val Loss: 0.8295, Val Macro F1: 0.7731


Epoch [12/30] - Train Loss: 0.6115, Macro F1: 0.9896 | Val Loss: 0.8623, Val Macro F1: 0.7843


Epoch [13/30] - Train Loss: 0.6095, Macro F1: 0.9906 | Val Loss: 0.8453, Val Macro F1: 0.7722

[Early Stopping] Model không cải thiện sau 12 epochs. Dừng huấn luyện.

[Thông báo] Khôi phục trọng số tốt nhất từ: model_oof/bilstm_oof_fold_3.pth


> Fold 3: Đã ghép thành công 617649 mẫu OOF vào meta_X_train_dl.

--- HOÀN TẤT PHA 1: CHUYỂN ĐỔI THÀNH DATAFRAME VÀ LƯU TRỮ MA TRẬN ĐẶC TRƯNG OOF ---
Đã lưu Meta Features OOF của Deep Learning tại: ip-files-train-meta/meta_X_train_dl.parquet
Lưu ý: Sẽ có những dòng mang giá trị NaN ở các fold (do bị triệt tiêu TIME_STEPS - 1 dòng đầu).


In [11]:
# --- GIAI ĐOẠN 2: HUẤN LUYỆN BASE MODEL CUỐI CÙNG TRÊN TOÀN BỘ TRAIN_DF ---
import gc
from sklearn.preprocessing import QuantileTransformer
from torch.utils.data import DataLoader
TIME_STEPS = 10 # Thay đổi giá trị này thành TIME_STEPS thực tế của bạn
BATCH_SIZE = 256
NUM_CLASSES = len(train_df['label'].unique())
MAX_SAMPLES_TOTAL = 20000
print("\n" + "="*50)
print("KHỞI ĐỘNG GIAI ĐOẠN 2: HUẤN LUYỆN FINAL BASE MODEL & SINH TEST OOF")
print("="*50)

# Gom rác xả các biến tạm khổng lồ của GĐ1
gc.collect()

# 1. TÁI SỬ DỤNG DataFrame gốc thay vì copy nhân bản (Tránh OOM)
# train_df, valid_df, test_df hiện tại sẽ trực tiếp nhận scale.
# Gọi reset_index để chắc chắn mảng chỉ mục chạy chuẩn xác.
full_train_df = train_df.reset_index(drop=True)
full_valid_df = valid_df.reset_index(drop=True)
full_test_df = test_df.reset_index(drop=True)

# 2. Áp dụng QuantileTransformer toàn cục trên 100% train_df
feat_cols = [col for col in full_train_df.columns if col != 'label']
scaler_full = QuantileTransformer(output_distribution="normal", random_state=42)

print("[1. Preprocessing] Đang fit_transform QuantileTransformer trên toàn bộ train_df...")
full_train_df[feat_cols] = scaler_full.fit_transform(full_train_df[feat_cols])
print("[1. Preprocessing] Đang transform valid_df và test_df...")
full_valid_df[feat_cols] = scaler_full.transform(full_valid_df[feat_cols])
full_test_df[feat_cols] = scaler_full.transform(full_test_df[feat_cols])

# 3. Tạo lại các Dataset
# Với train_df toàn phần, ta dùng full MAX_SAMPLES_TOTAL (20000)
print("\n[2. Dataset] Khởi tạo DataLoader cho cả 3 tập (Train, Valid, Test)...")
full_train_dataset = DynamicUndersampledSlidingWindowDataset(full_train_df, time_steps=TIME_STEPS, max_samples_per_class=MAX_SAMPLES_TOTAL, step=3)
full_valid_dataset = SequentialSlidingWindowDataset(full_valid_df, time_steps=10)
full_test_dataset = SequentialSlidingWindowDataset(full_test_df, time_steps=10)

full_train_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
full_valid_loader = DataLoader(full_valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
full_test_loader = DataLoader(full_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 4. Khởi tạo Model cuối cùng
print("\n[3. Modelling] Khởi tạo CNN-BiLSTM_Attention Final...")
final_model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
final_model_path = "model_oof/bilstm_oof_final_base.pth"

# 5. Huấn luyện (Early stopping dùng full_valid_loader)
print("\n[4. Training] Bắt đầu huấn luyện model Final Base...")
final_model = train_bi_lstm_cnn(
    model=final_model, 
    train_loader=full_train_loader, 
    val_loader=full_valid_loader, 
    device=DEVICE, 
    epochs=30, 
    patience=12, 
    save_path=final_model_path
)

# 6. Trích xuất đặc trưng OOF (Dự đoán) trên tập holdout (Valid và Test)
print("\n[5. Inference] Trích xuất Meta Features cho Validation và Test OOF...")
valid_preds = predict_bi_lstm_cnn(model=final_model, data_loader=full_valid_loader, device=DEVICE)
test_preds = predict_bi_lstm_cnn(model=final_model, data_loader=full_test_loader, device=DEVICE)

# 7. Ghép trả vào DataFrame và xử lý độ lệch TIME_STEPS
print("\n[6. Export] Khởi tạo ma trận NaN và gán lại độ lệch cửa sổ thời gian...")
meta_X_valid_dl = np.full((len(valid_df), NUM_CLASSES), np.nan)
meta_X_test_dl = np.full((len(test_df), NUM_CLASSES), np.nan)

# Index thực tế khi predict có cửa sổ = TIME_STEPS
valid_actual_idx = np.arange(TIME_STEPS - 1, len(valid_df))
test_actual_idx = np.arange(TIME_STEPS - 1, len(test_df))

meta_X_valid_dl[valid_actual_idx, :] = valid_preds
meta_X_test_dl[test_actual_idx, :] = test_preds

# Tạo DataFrame giữ nguyên Pandas Index gốc
dl_oof_columns = [f"dl_class_{i}" for i in range(NUM_CLASSES)]

meta_X_valid_dl_df = pd.DataFrame(meta_X_valid_dl, columns=dl_oof_columns, index=valid_df.index)
meta_X_valid_dl_df['label'] = valid_df['label'].values

meta_X_test_dl_df = pd.DataFrame(meta_X_test_dl, columns=dl_oof_columns, index=test_df.index)
meta_X_test_dl_df['label'] = test_df['label'].values

# Lưu Meta Features Valid và Test xuống ổ cứng
os.makedirs("ip-files-train-meta", exist_ok=True)
valid_save_path = "ip-files-train-meta/meta_X_valid_dl.parquet"
test_save_path = "ip-files-train-meta/meta_X_test_dl.parquet"

meta_X_valid_dl_df.to_parquet(valid_save_path, engine="pyarrow")
meta_X_test_dl_df.to_parquet(test_save_path, engine="pyarrow")

print(f"> Đã lưu Meta Features Valid tại: {valid_save_path}")
print(f"> Đã lưu Meta Features Test  tại: {test_save_path}")
print("\n=== HOÀN TẤT TOÀN BỘ PIPELINE STACKING CHO DEEP LEARNING (PHA 1 & PHA 2) ===")



KHỞI ĐỘNG GIAI ĐOẠN 2: HUẤN LUYỆN FINAL BASE MODEL & SINH TEST OOF
[1. Preprocessing] Đang fit_transform QuantileTransformer trên toàn bộ train_df...
[1. Preprocessing] Đang transform valid_df và test_df...

[2. Dataset] Khởi tạo DataLoader cho cả 3 tập (Train, Valid, Test)...
Class 0: 21496 cửa sổ
Class 1: 524407 cửa sổ
Class 2: 2721 cửa sổ
Class 3: 39271 cửa sổ
Class 4: 20556 cửa sổ
Class 5: 2652 cửa sổ
Class 6: 12830 cửa sổ
Class 7: 116384 cửa sổ
Class 8: 18140 cửa sổ
Class 9: 44977 cửa sổ
Class 10: 20109 cửa sổ
Khởi tạo Sequential Dataset:
- Chiều dài dữ liệu gốc: 570134
- Kích thước cửa sổ (time_steps): 10
- Số lượng mẫu (windows) sinh ra: 570125
-> Mẫu đầu tiên tương ứng với dòng index 9 của DataFrame.


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Khởi tạo Sequential Dataset:
- Chiều dài dữ liệu gốc: 760240
- Kích thước cửa sổ (time_steps): 10
- Số lượng mẫu (windows) sinh ra: 760231
-> Mẫu đầu tiên tương ứng với dòng index 9 của DataFrame.

[3. Modelling] Khởi tạo CNN-BiLSTM_Attention Final...

[4. Training] Bắt đầu huấn luyện model Final Base...


Epoch [1/30] - Train Loss: 0.9312, Macro F1: 0.8564 | Val Loss: 0.6172, Val Macro F1: 0.8982


Epoch [2/30] - Train Loss: 0.7347, Macro F1: 0.9469 | Val Loss: 0.5994, Val Macro F1: 0.9171


...

Epoch [27/30] - Train Loss: 0.5993, Macro F1: 0.9959 | Val Loss: 0.5813, Val Macro F1: 0.9443


Epoch [28/30] - Train Loss: 0.5964, Macro F1: 0.9967 | Val Loss: 0.5830, Val Macro F1: 0.9420


Epoch [29/30] - Train Loss: 0.5954, Macro F1: 0.9969 | Val Loss: 0.5776, Val Macro F1: 0.9425


Epoch [30/30] - Train Loss: 0.5946, Macro F1: 0.9969 | Val Loss: 0.5953, Val Macro F1: 0.9251

[Thông báo] Khôi phục trọng số tốt nhất từ: model_oof/bilstm_oof_final_base.pth

[5. Inference] Trích xuất Meta Features cho Validation và Test OOF...



[6. Export] Khởi tạo ma trận NaN và gán lại độ lệch cửa sổ thời gian...
> Đã lưu Meta Features Valid tại: ip-files-train-meta/meta_X_valid_dl.parquet
> Đã lưu Meta Features Test  tại: ip-files-train-meta/meta_X_test_dl.parquet

=== HOÀN TẤT TOÀN BỘ PIPELINE STACKING CHO DEEP LEARNING (PHA 1 & PHA 2) ===


In [12]:
# test model trên tập test để xem kết quả
final_model.eval()
all_test_preds = []
with torch.no_grad():
    for inputs, _ in tqdm(full_test_loader, desc="[Đang dự đoán trên Test]", leave=False):
        inputs = inputs.to(DEVICE)
        outputs = final_model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.append(predicted.cpu().numpy())
all_test_preds = np.concatenate(all_test_preds)
print("\nClassification Report trên Test Set:")
print(classification_report(test_df['label'].values[TIME_STEPS - 1 :], all_test_preds, digits=4))



Classification Report trên Test Set:
              precision    recall  f1-score   support

           0     0.8952    0.9880    0.9393     19846
           1     0.9996    1.0000    0.9998    484077
           2     0.5803    0.9980    0.7339      2515
           3     0.9977    0.9340    0.9648     36253
           4     0.5988    0.8802    0.7127     18979
           5     0.9959    0.9971    0.9965      2451
           6     0.7302    0.7051    0.7174     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.8548    0.9882    0.9167     16746
           9     1.0000    0.6762    0.8068     41514
          10     0.9639    0.9927    0.9781     18567

    accuracy                         0.9708    760231
   macro avg     0.8742    0.9236    0.8878    760231
weighted avg     0.9772    0.9708    0.9712    760231



In [ ]:
#load thử model model_final/best_cnn_bilstm_best.pth và dự đoán trên test_df
model_before = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
model_before.load_state_dict(torch.load(r"model_final\best_cnn_bilstm_best.pth", map_location=DEVICE, weights_only=True))
model_before.eval()
all_test_preds_before = []
with torch.no_grad():
    for inputs, _ in tqdm(full_test_loader, desc="[Dự đoán trên Test với model cũ]", leave=False):
        inputs = inputs.to(DEVICE)
        outputs = model_before(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds_before.append(predicted.cpu().numpy())
all_test_preds_before = np.concatenate(all_test_preds_before)
print("\nClassification Report trên Test Set với model cũ:")
print(classification_report(test_df['label'].values[TIME_STEPS - 1 :], all_test_preds_before, digits=4))


Classification Report trên Test Set với model cũ:
              precision    recall  f1-score   support

           0     0.7348    0.9931    0.8447     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.6413    0.9968    0.7805      2515
           3     0.9994    0.9561    0.9773     36253
           4     0.5502    0.7172    0.6227     18979
           5     0.9955    0.9980    0.9967      2451
           6     0.7667    0.7095    0.7370     11847
           7     1.0000    0.9999    0.9999    107436
           8     0.6961    0.7622    0.7277     16746
           9     0.9999    0.6731    0.8045     41514
          10     0.9722    0.9880    0.9800     18567

    accuracy                         0.9627    760231
   macro avg     0.8506    0.8903    0.8610    760231
weighted avg     0.9696    0.9627    0.9634    760231



In [11]:
# tạo test_df cũ để xem kết quả
test_df_old = pd.read_parquet(r"final_data\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")

# 1. Tái tạo lại timestamp_num để sort như cũ
if 'timestamp' in test_df_old.columns:
    test_df_old["timestamp_num"] = pd.to_datetime(test_df_old['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
    
# 2. Sort lại toàn bộ test theo thời gian
if 'timestamp_num' in test_df_old.columns:
    test_df_old.sort_values(by='timestamp_num', inplace=True)

# 3. Drop cột rác (kể cả timestamp_num sau khi sort xong)
test_df_old.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# 4. Khởi tạo Dataset dùng chung class UndersampledSlidingWindowDataset (kèm shuffle nội bộ của class này nảy sinh)
test_dataset_old = UndersampledSlidingWindowDataset(test_df_old, TIME_STEPS, max_samples_per_class=10000000, step=1)
test_loader_old = DataLoader(test_dataset_old, batch_size=BATCH_SIZE, shuffle=False)

# 5. Khởi tạo model và load trọng số
model_old_verify = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
model_old_verify.load_state_dict(torch.load(r"model_final\best_cnn_bilstm_best.pth", map_location=DEVICE, weights_only=True))
model_old_verify.eval()

# 6. Dự đoán và in Classification Report
all_test_preds_old = []
all_test_targets_old = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader_old, desc="[Verify test_df cũ với model cũ]", leave=False):
        inputs = inputs.to(DEVICE)
        outputs = model_old_verify(inputs)
        _, predicted = torch.max(outputs.data, 1)
        
        all_test_preds_old.append(predicted.cpu().numpy())
        all_test_targets_old.append(labels.numpy())

all_test_preds_old = np.concatenate(all_test_preds_old)
all_test_targets_old = np.concatenate(all_test_targets_old)

print("\nClassification Report trên test_df_prepared.parquet (Cũ):")
print(classification_report(all_test_targets_old, all_test_preds_old, digits=4))

Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 19846 cửa sổ (step=1).
Class 1: Giữ nguyên 484077 cửa sổ (step=1).
Class 2: Giữ nguyên 2515 cửa sổ (step=1).
Class 3: Giữ nguyên 36253 cửa sổ (step=1).
Class 4: Giữ nguyên 18979 cửa sổ (step=1).
Class 5: Giữ nguyên 2451 cửa sổ (step=1).
Class 6: Giữ nguyên 11847 cửa sổ (step=1).
Class 7: Giữ nguyên 107436 cửa sổ (step=1).
Class 8: Giữ nguyên 16746 cửa sổ (step=1).
Class 9: Giữ nguyên 41514 cửa sổ (step=1).
Class 10: Giữ nguyên 18567 cửa sổ (step=1).
Tổng số cửa sổ sau khi lọc và Undersampling: 760231



Classification Report trên test_df_prepared.parquet (Cũ):
              precision    recall  f1-score   support

           0     0.8995    0.9746    0.9355     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.7049    0.9964    0.8257      2515
           3     0.9986    0.9667    0.9824     36253
           4     0.5712    0.8541    0.6845     18979
           5     0.9971    0.9980    0.9976      2451
           6     0.7594    0.7072    0.7324     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.8241    0.9749    0.8932     16746
           9     0.9999    0.6769    0.8073     41514
          10     0.9788    0.9901    0.9844     18567

    accuracy                         0.9711    760231
   macro avg     0.8849    0.9217    0.8948    760231
weighted avg     0.9775    0.9711    0.9716    760231



In [12]:
# test model mới trên test_df_old để xem kết quả
model_new_verify = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
model_new_verify.load_state_dict(torch.load(r"model_oof\bilstm_oof_final_base.pth", map_location=DEVICE, weights_only=True))
model_new_verify.eval()
all_test_preds_new = []
with torch.no_grad():
    for inputs, _ in tqdm(test_loader_old, desc="[Verify test_df cũ với model mới]", leave=False):
        inputs = inputs.to(DEVICE)
        outputs = model_new_verify(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds_new.append(predicted.cpu().numpy())
all_test_preds_new = np.concatenate(all_test_preds_new)
print("\nClassification Report trên test_df_prepared.parquet với model mới:")
print(classification_report(all_test_targets_old, all_test_preds_new, digits=4))

[Verify test_df cũ với model mới]:   0%|          | 0/2970 [00:00<?, ?it/s]


Classification Report trên test_df_prepared.parquet với model mới:
              precision    recall  f1-score   support

           0     0.9346    0.7727    0.8460     19846
           1     0.9963    1.0000    0.9981    484077
           2     0.5534    0.9976    0.7119      2515
           3     1.0000    0.8960    0.9452     36253
           4     0.5003    0.8671    0.6345     18979
           5     0.9992    0.9796    0.9893      2451
           6     0.8382    0.6949    0.7598     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.6998    0.9962    0.8221     16746
           9     1.0000    0.6274    0.7710     41514
          10     0.9904    0.9870    0.9887     18567

    accuracy                         0.9602    760231
   macro avg     0.8647    0.8926    0.8606    760231
weighted avg     0.9726    0.9602    0.9616    760231

